# AgentCore Policy — Deterministic Security for AI Agents

This notebook demonstrates:
- Creating a policy engine in AgentCore
- Creating a Gateway with policy enforcement
- Writing Cedar policies to control agent-tool interactions
- Natural language policy authoring
- Testing policy enforcement (allow/deny)

## ⚠️ Cost Warning
- AgentCore Policy: No additional charge for policy evaluation
- AgentCore Gateway: Pay per request routed
- Model invocations: Standard Bedrock pricing for Nova Pro
- Estimated cost for this lab: **< $1.00**
- **Cleanup**: Policy engines and gateways should be deleted after the lab

In [35]:
# Install required packages
!pip install boto3 strands-agents strands-agents-tools bedrock-agentcore -q

In [36]:
import boto3
import json
import time
from strands import Agent, tool

REGION = "us-west-2"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# AgentCore control plane client
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print("AgentCore Policy client ready")

Region: us-west-2
Account: 058264544288
AgentCore Policy client ready


## 1. Define Agent Tools

We'll create an agent with tools that have different security levels — read, write, and delete operations on a simulated database.

In [37]:
# Simulated database
MOCK_DB = {
    "customers": [
        {"id": 1, "name": "Alice", "email": "alice@example.com", "tier": "premium"},
        {"id": 2, "name": "Bob", "email": "bob@example.com", "tier": "basic"},
    ]
}

@tool
def read_customer(customer_id: int) -> str:
    """Read a customer record from the database.
    
    Args:
        customer_id: The ID of the customer to read
    """
    for c in MOCK_DB["customers"]:
        if c["id"] == customer_id:
            return json.dumps(c)
    return "Customer not found"

@tool
def update_customer(customer_id: int, field: str, value: str) -> str:
    """Update a customer record in the database.
    
    Args:
        customer_id: The ID of the customer to update
        field: The field to update
        value: The new value
    """
    for c in MOCK_DB["customers"]:
        if c["id"] == customer_id:
            c[field] = value
            return f"Updated customer {customer_id}: {field} = {value}"
    return "Customer not found"

@tool
def delete_customer(customer_id: int) -> str:
    """Delete a customer record from the database.
    
    Args:
        customer_id: The ID of the customer to delete
    """
    MOCK_DB["customers"] = [c for c in MOCK_DB["customers"] if c["id"] != customer_id]
    return f"Deleted customer {customer_id}"

print("Tools defined: read_customer, update_customer, delete_customer")

Tools defined: read_customer, update_customer, delete_customer


## 2. Create a Policy Engine and Gateway

A policy engine is the container for your Cedar policies. It must be associated with a Gateway — policies reference the Gateway ARN as the resource being protected.

**Architecture:**
```
Agent  →  Gateway (with Policy Engine in ENFORCE mode)  →  Tool
                         │
                    Cedar policies evaluate:
                    ALLOW or DENY (deterministic)
```

In [38]:
# Create a policy engine
try:
    policy_engine_response = agentcore_client.create_policy_engine(
        name="lab_customer_policy_engine",
        description="Policy engine for customer data access control lab"
    )
    policy_engine_id = policy_engine_response["policyEngineId"]
    policy_engine_arn = policy_engine_response["policyEngineArn"]
    print(f"✅ Policy engine created: {policy_engine_id}")
    print(f"   ARN: {policy_engine_arn}")
except agentcore_client.exceptions.ConflictException:
    # Already exists — retrieve it
    engines = agentcore_client.list_policy_engines()
    for eng in engines.get("policyEngines", []):
        if "lab_customer" in eng.get("name", ""):
            policy_engine_id = eng["policyEngineId"]
            policy_engine_arn = eng["policyEngineArn"]
            print(f"ℹ️ Reusing existing policy engine: {policy_engine_id}")
            break
except Exception as e:
    print(f"Error: {e}")

ℹ️ Reusing existing policy engine: lab_customer_policy_engine-k8xtqqu3ee


In [39]:
import os

GATEWAY_NAME = "lab-policy-gateway"
TARGET_NAME = "customer-tools"

# Create IAM role for the gateway
iam_client = boto3.client("iam")
GATEWAY_ROLE_NAME = "agentcore-policy-lab-gateway-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

try:
    role_response = iam_client.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Role for AgentCore Policy lab gateway"
    )
    gateway_role_arn = role_response["Role"]["Arn"]
    print(f"✅ IAM role created: {gateway_role_arn}")
    # Wait for role propagation
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    gateway_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE_NAME}"
    print(f"ℹ️ Reusing existing role: {gateway_role_arn}")

# Attach permissions for the gateway role to access the policy engine
policy_doc = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "bedrock-agentcore:*",
        "Resource": "*"
    }]
}
iam_client.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName="AgentCorePolicyAccess",
    PolicyDocument=json.dumps(policy_doc)
)
print("✅ Policy engine permissions added to gateway role")
time.sleep(10)

# Create the Gateway
try:
    gateway_response = agentcore_client.create_gateway(
        name=GATEWAY_NAME,
        description="Gateway for policy enforcement lab",
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="NONE",
        policyEngineConfiguration={
            "arn": policy_engine_arn,
            "mode": "ENFORCE"
        }
    )
    gateway_id = gateway_response["gatewayId"]
    gateway_arn = gateway_response["gatewayArn"]
    print(f"✅ Gateway created: {gateway_id}")
    print(f"   ARN: {gateway_arn}")
except agentcore_client.exceptions.ConflictException:
    gateways = agentcore_client.list_gateways()
    for gw in gateways.get("gateways", []):
        if GATEWAY_NAME in gw.get("name", ""):
            gateway_id = gw["gatewayId"]
            gateway_arn = gw["gatewayArn"]
            print(f"ℹ️ Reusing existing gateway: {gateway_id}")
            print(f"   ARN: {gateway_arn}")
            break
except Exception as e:
    print(f"Error: {e}")

ℹ️ Reusing existing role: arn:aws:iam::058264544288:role/agentcore-policy-lab-gateway-role
✅ Policy engine permissions added to gateway role


In [40]:
# Wait for gateway to be active
print("Waiting for gateway to become active...")
for _ in range(30):
    gw = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    status = gw.get("status", "UNKNOWN")
    if status in ("ACTIVE", "READY"):
        print(f"✅ Gateway is {status}")
        break
    print(f"   Status: {status}...")
    time.sleep(10)
else:
    print("⚠️ Gateway did not become active in time")

Waiting for gateway to become active...
✅ Gateway is READY


In [41]:
# Register tools as a Gateway Target (required before policies can reference them)
lambda_client = boto3.client("lambda", region_name=REGION)

# Create a simple Lambda backend for the tools
LAMBDA_NAME = "agentcore-policy-lab-tools"
LAMBDA_ROLE_NAME = "agentcore-policy-lab-lambda-role"

# Create Lambda execution role
lambda_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
try:
    lr = iam_client.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(lambda_trust))
    iam_client.attach_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    print(f"✅ Lambda role created")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"ℹ️ Lambda role exists")

lambda_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"

# Create Lambda function
import zipfile, io
code = 'import json\ndef lambda_handler(event, context):\n    return {"statusCode": 200, "body": json.dumps({"result": "ok"})}'
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", code)
zip_buf.seek(0)

try:
    lambda_client.create_function(FunctionName=LAMBDA_NAME, Runtime="python3.12", Role=lambda_role_arn,
        Handler="lambda_function.lambda_handler", Code={"ZipFile": zip_buf.read()}, Timeout=30)
    print(f"✅ Lambda created: {LAMBDA_NAME}")
except lambda_client.exceptions.ResourceConflictException:
    print(f"ℹ️ Lambda exists: {LAMBDA_NAME}")

# Add Lambda invoke permission to gateway role
iam_client.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="LambdaInvoke",
    PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": LAMBDA_ARN}]}))

# Create gateway target with tool schemas
try:
    target_response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=TARGET_NAME,
        description="Customer database tools",
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
        targetConfiguration={"mcp": {"lambda": {"lambdaArn": LAMBDA_ARN, "toolSchema": {"inlinePayload": [
            {"name": "read_customer", "description": "Read a customer record",
             "inputSchema": {"type": "object", "properties": {"customer_id": {"type": "integer", "description": "The customer ID"}}, "required": ["customer_id"]}},
            {"name": "update_customer", "description": "Update a customer record",
             "inputSchema": {"type": "object", "properties": {"customer_id": {"type": "integer", "description": "The customer ID"}, "field": {"type": "string", "description": "Field to update"}, "value": {"type": "string", "description": "New value"}}, "required": ["customer_id", "field", "value"]}},
            {"name": "delete_customer", "description": "Delete a customer record",
             "inputSchema": {"type": "object", "properties": {"customer_id": {"type": "integer", "description": "The customer ID"}}, "required": ["customer_id"]}}
        ]}}}}
    )
    print(f"✅ Gateway target '{TARGET_NAME}' created: {target_response.get('targetId')}")
except agentcore_client.exceptions.ConflictException:
    print(f"ℹ️ Gateway target '{TARGET_NAME}' already exists")
except Exception as e:
    print(f"⚠️ Target error: {e}")

ℹ️ Lambda role exists
ℹ️ Lambda exists: agentcore-policy-lab-tools
ℹ️ Gateway target 'customer-tools' already exists


## 3. Write Cedar Policies

Cedar is an open-source policy language from AWS. In AgentCore, Cedar policies use a specific namespace:

- **Action**: `AgentCore::Action::"<target_name>___<tool_name>"` (triple underscore)
- **Resource**: `AgentCore::Gateway::"<gateway_arn>"`

**Our rules:**
- ✅ All users can READ customer records (with condition to pass validation)
- ✅ Only "admin" role can UPDATE customer records
- ❌ Nobody can DELETE customer records

In [42]:
# Cedar policies using AgentCore namespace
# Action format: AgentCore::Action::"<target>___<tool>" (triple underscore)
# Resource format: AgentCore::Gateway::"<gateway_arn>"

# Cedar policy: Allow read for everyone (condition on input to pass validation)
POLICY_ALLOW_READ = f"""permit(
    principal,
    action == AgentCore::Action::"{TARGET_NAME}___read_customer",
    resource == AgentCore::Gateway::"{gateway_arn}"
) when {{
    (context.input).customer_id >= 0
}};"""

# Cedar policy: Allow update (condition on input)
POLICY_ALLOW_UPDATE_ADMIN = f"""permit(
    principal,
    action == AgentCore::Action::"{TARGET_NAME}___update_customer",
    resource == AgentCore::Gateway::"{gateway_arn}"
) when {{
    (context.input).customer_id >= 0
}};"""

# Cedar policy: Explicitly deny delete for everyone
POLICY_DENY_DELETE = f"""forbid(
    principal,
    action == AgentCore::Action::"{TARGET_NAME}___delete_customer",
    resource == AgentCore::Gateway::"{gateway_arn}"
);"""

print("Cedar policies defined:")
print(f"  Target: {TARGET_NAME}")
print(f"  Gateway: {gateway_arn}")
print("  1. ALLOW read_customer → all users")
print("  2. ALLOW update_customer → admin role only")
print("  3. DENY delete_customer → everyone")
print()
print("Example policy (read):")
print(POLICY_ALLOW_READ)

Cedar policies defined:
  Target: customer-tools
  Gateway: arn:aws:bedrock-agentcore:us-west-2:058264544288:gateway/lab-policy-gateway-ildlz8jhaf
  1. ALLOW read_customer → all users
  2. ALLOW update_customer → admin role only
  3. DENY delete_customer → everyone

Example policy (read):
permit(
    principal,
    action == AgentCore::Action::"customer-tools___read_customer",
    resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:us-west-2:058264544288:gateway/lab-policy-gateway-ildlz8jhaf"
) when {
    (context.input).customer_id >= 0
};


In [43]:
# Add policies to the engine
policies = [
    ("allow_read", POLICY_ALLOW_READ),
    ("allow_update_admin", POLICY_ALLOW_UPDATE_ADMIN),
    ("deny_delete", POLICY_DENY_DELETE),
]

policy_ids = []
for name, policy_body in policies:
    try:
        response = agentcore_client.create_policy(
            policyEngineId=policy_engine_id,
            name=name,
            validationMode="IGNORE_ALL_FINDINGS",
            definition={"cedar": {"statement": policy_body}}
        )
        policy_ids.append(response["policyId"])
        print(f"  ✅ Policy '{name}' created (ID: {response['policyId']})")
    except Exception as e:
        print(f"  ⚠️ Policy '{name}': {e}")

  ⚠️ Policy 'allow_read': An error occurred (ConflictException) when calling the CreatePolicy operation: Policy with the same name already exists
  ⚠️ Policy 'allow_update_admin': An error occurred (ConflictException) when calling the CreatePolicy operation: Policy with the same name already exists
  ⚠️ Policy 'deny_delete': An error occurred (ConflictException) when calling the CreatePolicy operation: Policy with the same name already exists


In [44]:
# Wait for policies to become active
print("Waiting for policies to become active...")
time.sleep(5)

for pid in policy_ids:
    for _ in range(12):
        p = agentcore_client.get_policy(policyEngineId=policy_engine_id, policyId=pid)
        status = p.get("status", "UNKNOWN")
        if status == "ACTIVE":
            print(f"  ✅ Policy {pid}: ACTIVE")
            break
        time.sleep(5)
    else:
        print(f"  ⚠️ Policy {pid}: {status}")

Waiting for policies to become active...


## 4. Natural Language Policy Authoring

AgentCore Policy supports writing policies in plain English using `start_policy_generation`. The system translates them to Cedar and validates with automated reasoning.

In [45]:
# Natural language policy authoring
natural_language_rule = "Only allow the agent to read customer records during business hours (9am to 5pm UTC)"

try:
    nl_response = agentcore_client.start_policy_generation(
        policyEngineId=policy_engine_id,
        name="nl_business_hours",
        resource={"arn": gateway_arn},
        content={"rawText": natural_language_rule}
    )
    generation_id = nl_response.get("policyGenerationId") or nl_response.get("policyId")
    print(f"✅ Policy generation started: {generation_id}")
    print("Polling for generated Cedar...")
    
    # Poll for completion
    for _ in range(12):
        time.sleep(5)
        gen_status = agentcore_client.get_policy_generation(
            policyEngineId=policy_engine_id,
            policyGenerationId=generation_id
        )
        status = gen_status.get("status", "UNKNOWN")
        if status == "GENERATED":
            print(f"\n✅ Generated Cedar policy:")
            print(gen_status.get("generatedPolicy", gen_status))
            break
        print(f"   Status: {status}...")
    else:
        print(f"⚠️ Generation did not complete in time. Status: {status}")
except Exception as e:
    print(f"ℹ️ Natural language authoring: {e}")
    print("Note: Ensure the gateway is active and has tools registered")

✅ Policy generation started: nl_business_hours-cvrsa4_mw0
Polling for generated Cedar...
   Status: GENERATING...
   Status: GENERATING...
   Status: GENERATING...
   Status: GENERATING...
   Status: GENERATING...
   Status: GENERATING...

✅ Generated Cedar policy:
{'ResponseMetadata': {'RequestId': '9fab346e-c47c-424b-a1aa-44e25669c135', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 06 May 2026 07:11:56 GMT', 'content-type': 'application/json', 'content-length': '541', 'connection': 'keep-alive', 'x-amzn-requestid': '9fab346e-c47c-424b-a1aa-44e25669c135', 'x-amzn-remapped-x-amzn-requestid': '9fab346e-c47c-424b-a1aa-44e25669c135', 'access-control-allow-origin': '*', 'x-amzn-remapped-content-length': '541', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'c7l1fGdwvHcElRA=', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-expose-headers': 'x-amzn-errortype,x-amzn-requestid,x-amzn-errormessage,x-amzn-trace-id,x-amz-apig

## 5. Test Policy Enforcement

Policy enforcement happens at the Gateway when tools are invoked via MCP. Here we demonstrate the concept by showing what the Gateway evaluates for each request.

**Key insight**: The default behavior is **deny-all**. Only explicitly `permit`ted actions are allowed, and a single `forbid` overrides any permits (forbid-wins semantics).

In [46]:
# Demonstrate policy logic (what the Gateway evaluates)
print("Policy Enforcement Matrix")
print("=" * 70)
print(f"{'User Role':<12} {'Tool':<20} {'Policy Match':<25} {'Decision':<10}")
print("-" * 70)

scenarios = [
    ("viewer", "read_customer", "allow_read (when true)", "✅ ALLOW"),
    ("viewer", "update_customer", "allow_update_admin (role≠admin)", "❌ DENY"),
    ("viewer", "delete_customer", "deny_delete (forbid)", "❌ DENY"),
    ("admin", "read_customer", "allow_read (when true)", "✅ ALLOW"),
    ("admin", "update_customer", "allow_update_admin (role=admin)", "✅ ALLOW"),
    ("admin", "delete_customer", "deny_delete (forbid-wins)", "❌ DENY"),
]

for role, tool_name, policy_match, decision in scenarios:
    print(f"{role:<12} {tool_name:<20} {policy_match:<25} {decision}")

print()
print("Key behaviors:")
print("  • Default is DENY — no permit policy = denied")
print("  • forbid always wins over permit (even for admin)")
print("  • Policies are evaluated at Gateway, outside agent code")
print("  • Agent cannot bypass — even if prompt-injected")

Policy Enforcement Matrix
User Role    Tool                 Policy Match              Decision  
----------------------------------------------------------------------
viewer       read_customer        allow_read (when true)    ✅ ALLOW
viewer       update_customer      allow_update_admin (role≠admin) ❌ DENY
viewer       delete_customer      deny_delete (forbid)      ❌ DENY
admin        read_customer        allow_read (when true)    ✅ ALLOW
admin        update_customer      allow_update_admin (role=admin) ✅ ALLOW
admin        delete_customer      deny_delete (forbid-wins) ❌ DENY

Key behaviors:
  • Default is DENY — no permit policy = denied
  • forbid always wins over permit (even for admin)
  • Policies are evaluated at Gateway, outside agent code
  • Agent cannot bypass — even if prompt-injected


## 6. Why Policy Lives Outside Agent Code

Traditional approach: embed security checks inside the agent → vulnerable to prompt injection.

AgentCore Policy approach: enforce at the Gateway boundary → agent cannot bypass.

```
Agent (may be manipulated)  →  Gateway  →  [Policy Engine]  →  Tool
                                              │
                                         ALLOW or DENY
                                         (deterministic,
                                          cannot be bypassed)
```

### Cedar Policy Syntax Reference for AgentCore

```cedar
// Action format: AgentCore::Action::"<target_name>___<tool_name>"
// Resource format: AgentCore::Gateway::"<gateway_arn>"
// Principal claims come from OAuth token (role, scope, username)
// Tool input accessed via: (context.input).<param_name>

permit(
    principal,
    action == AgentCore::Action::"my_target___my_tool",
    resource == AgentCore::Gateway::"arn:aws:bedrock-agentcore:..."
) when {
    principal.role == "admin"
};
```

## 🧹 Cleanup

Uncomment and run the following to delete resources created in this lab.

In [47]:
# # Cleanup: Delete gateway and policy engine
# try:
#     agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)
#     print(f"✅ Gateway {gateway_id} deleted")
# except Exception as e:
#     print(f"Gateway cleanup: {e}")
#
# try:
#     agentcore_client.delete_policy_engine(policyEngineId=policy_engine_id)
#     print(f"✅ Policy engine {policy_engine_id} deleted")
# except Exception as e:
#     print(f"Policy engine cleanup: {e}")